# ReAct Framework: Reasoning and Acting in Large Language Models

> ReACT is a framework that merges **thoughtful planning (reasoning)** with **external tool use (action)**.

### 🔄 How It Works:
1. **Initial Thought**: The model explains what it plans to do.
2. **Action**: Executes a tool-based task (e.g., search Wikipedia).
3. **Observation**: Receives the result of that action.
4. **Refined Thought**: Updates its reasoning.
5. **Final Answer**: Produces a confident, evidence-backed output.

## CoT - Chain of thought Reasoning

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
print(llm)

client=<openai.resources.chat.completions.completions.Completions object at 0x10bcab8f0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10c683800> root_client=<openai.OpenAI object at 0x10b8c0b30> root_async_client=<openai.AsyncOpenAI object at 0x10bcab9b0> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********')


In [3]:
text = "Exaplain step by step. How old is the president of the United States?"

result = llm.invoke(text)
print(result)

content="As of my last update, the President of the United States is Joe Biden. He was born on November 20, 1942. To determine his current age, you would follow these steps:\n\n1. **Identify the Current Year:** Determine the current year. For example, let's use 2023.\n   \n2. **Subtract the Birth Year from the Current Year:**\n   - 2023 (current year) - 1942 (birth year) = 81\n\n3. **Check If the Birthday Has Passed This Year:**\n   - Joe Biden's birthday is on November 20th.\n   - If today's date is before November 20, he hasn't had his birthday yet this year, so he is still 80 years old.\n   - If today's date is on or after November 20, he has turned 81 years old.\n\nTherefore, Joe Biden would be 80 years old if today is before November 20, 2023, and 81 years old if today is on or after November 20, 2023." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 214, 'prompt_tokens': 24, 'total_tokens': 238, 'completion_tokens_details': {'accepted_p

In [4]:
## stroutput Parser

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Reason for the answers based on the questions",
        ),
        ("user", "{input}"),
    ]
)

output_parser = StrOutputParser()
chain = prompt | llm | output_parser

response = chain.invoke({"input": text})
print(response)


As of October 2023, the President of the United States is Joe Biden. Here's how you can determine his age step by step:

1. **Identify Birth Date**: Joe Biden was born on November 20, 1942.
   
2. **Calculate Age**: To find out how old he is as of October 2023, you need to subtract the birth year from the current year.
   - Current year: 2023
   - Birth year: 1942
   - Subtract these numbers: 2023 - 1942 = 81

3. **Consider Birth Month and Day**: Since we are calculating his age in October 2023, and his birthday is in November, he has not yet turned 81 this year. 
   
4. **Conclude**: Therefore, as of October 2023, Joe Biden is 80 years old, and he will turn 81 in November 2023.


### Using Tools

In [5]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [6]:
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [7]:
tools = [wikipedia]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm = ChatOpenAI(model="gpt-4o", temperature=0)

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

question = "How old is the president of the United States?"

result = agent_executor.invoke(
    {"input": question, "chat_history": []}
)
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'President of the United States'}`


Page: President of the United States
Summary: The president of the United States (POTUS) is the head of state and head of government of the United States. The president directs the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces.
The power of the presidency has grown since the first president, George Washington, took office in 1789. While presidential power has ebbed and flowed over time, the presidency has played an increasing role in American political life since the beginning of the 20th century, carrying over into the 21st century with some expansions during the presidencies of Franklin D. Roosevelt and George W. Bush. In modern times, the president is one of the world's most powerful political figures and the leader of the world's only remaining superpower. As the leader of the nation with the largest economy

### Multi hop

In [8]:
import langchain

langchain.debug = True

In [9]:
question = "What is the first film that Russel Crowe won an Oscar for, and who directed that movie?"

result = agent_executor.invoke({"input": question, "chat_history": []})
print(result["output"])

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "What is the first film that Russel Crowe won an Oscar for, and who directed that movie?",
  "chat_history": []
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad>] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scratchpad>] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scratchpad> > chain:RunnableLambda] Entering Chain run with input:
{
  "input": ""
}
[chain/end] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scra